# 04b — B0: scale-matched QC classifier test (`cultureQC_upgrade_specv3.md` §2B.1)

**Question.** Does resizing C2C12 frames to the training tiles' physical scale before the 256 px tile is cut
make the QC classifier call real frames correctly? Today it calls 5.0% of held-out normal C2C12 frames "normal"
and 67.1% "image_quality" (`results/classifier_c2c12.md`).

**Scale** (`configs/qc.yaml`, both from the papers' Methods): C2C12 1.3 µm/px, LIVECell 1.243 µm/px, so
f = 1.046. That is close to 1, so a large effect is not expected; the test is run as specified either way.

**Protocol** (fixed in `scripts/classifier_scale_test.py` before this run): choose the multiplier from
f × {0.75, 1, 1.25} on the **tuning** fleet only; report **held-out** once for unscaled and the chosen one.
Pass = normal→normal ≥ 80%, contamination recall ≥ 85%, normal→contamination ≤ 5%. Fail → B3 retrain (`nb/05`).

**I/O rule.** Nothing reads frames through the Drive mount. The notebook copies **one archive**
(`staged/c2c12_prepared.zip`, written by `nb/03`) and **one table** (`cache/logits.parquet`, for the parity
check) to local disk, works there, and copies **one results zip** back to Drive.

**Runtime.** No Cellpose, classifier only: ~2,700 frames, 6 variants on tuning and 2 on held-out. A GPU runtime
(T4 is plenty) makes it minutes; CPU works but is slower (not measured).

C2C12: Ker et al., *Sci Data* 5:180237 (2018), OSF `ysaq2`, **CC BY 4.0**. Faults are **simulated**.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, time
DRIVE_ROOT = '/content/drive/MyDrive/cultureqc'
STAGE_DIR = f'{DRIVE_ROOT}/staged'
PREPARED_ZIP = f'{STAGE_DIR}/c2c12_prepared.zip'       # nb/03: C2C12 + fault PNGs, frames CSVs, fault manifest
CACHED_LOGITS = f'{DRIVE_ROOT}/cache/logits.parquet'   # qc_effnetb0_v1 logits, for the parity check
RESULTS_ZIP = f'{STAGE_DIR}/b0_classifier_scale_results.zip'
LOCAL_ROOT = '/content/data'
C2C12_DIR = f'{LOCAL_ROOT}/c2c12'
FAULT_DIR = f'{LOCAL_ROOT}/c2c12_faults'
OUT_DIR = '/content/b0_results'
for p in (PREPARED_ZIP, CACHED_LOGITS):
    assert os.path.exists(p), f'missing on Drive: {p}'
print(f'{PREPARED_ZIP}: {os.path.getsize(PREPARED_ZIP)/1e9:.2f} GB')

In [ ]:
REPO_URL = 'https://github.com/n1tishc/cultureqc.git'
BRANCH = 'slice-1b-compute-cache'
PINNED_SHA = 'a0af827'   # B0: scripts/classifier_scale_test.py + configs/qc.yaml + culture.qc rescale helpers

!rm -rf /content/cultureqc
!git clone --branch $BRANCH $REPO_URL /content/cultureqc
%cd /content/cultureqc
!git checkout $PINNED_SHA
!git rev-parse HEAD
for f in ('scripts/classifier_scale_test.py', 'configs/qc.yaml', 'results/replay_fleet_split.csv'):
    assert os.path.exists(f), f'{f} missing at this commit'

In [ ]:
!pip install -q -r requirements.txt
import torch
print('torch', torch.__version__, '— CUDA available:', torch.cuda.is_available(), '(not required; faster)')

## 1 — Bring the data to local disk (archive copy, then unzip locally)

`nb/03` zipped from `/`, so unzipping at `/` restores the `/content/data/...` paths its CSVs point at.

In [ ]:
t0 = time.time()
local_zip = '/content/c2c12_prepared.zip'
shutil.copy(PREPARED_ZIP, local_zip)                   # one sequential read from Drive
!unzip -q -o $local_zip -d /
os.remove(local_zip)
shutil.copy(CACHED_LOGITS, '/content/cached_logits.parquet')
print(f'copied + unzipped in {(time.time()-t0)/60:.1f} min')
for p in (f'{C2C12_DIR}/c2c12_frames.csv', f'{FAULT_DIR}/fault_frames.csv', f'{FAULT_DIR}/fault_manifest.parquet'):
    assert os.path.exists(p), p
!du -sh $C2C12_DIR/png $FAULT_DIR

## 2 — Frame counts (no inference)

Must match Phase A exactly: tuning 860 normal / 48 contamination / 225 dimming; held-out 1228 / 49 / 327.
The Drive copy of the fault manifest predates `3d0c140`; the script's selection rule gives the same frames
with either version (checked on the Mac), and it verifies every fault PNG's sha against the manifest.

In [ ]:
!python scripts/classifier_scale_test.py --c2c12-dir $C2C12_DIR --fault-dir $FAULT_DIR --list-only

### ⏸ Checkpoint — do the counts above match?

If not, stop and report the table; don't run step 3.

## 3 — Run B0 (tuning → choice → held-out once)

In [ ]:
t0 = time.time()
!python scripts/classifier_scale_test.py --c2c12-dir $C2C12_DIR --fault-dir $FAULT_DIR \
    --cached-logits /content/cached_logits.parquet --out $OUT_DIR
print(f'B0 run: {(time.time()-t0)/60:.1f} min')

In [ ]:
from IPython.display import Image, display
display(Image(f'{OUT_DIR}/classifier_scale_examples.png'))

## 4 — Results back to Drive (one zip)

In [ ]:
!cd /content && zip -q -r $RESULTS_ZIP b0_results
print(RESULTS_ZIP, f'{os.path.getsize(RESULTS_ZIP)/1e6:.1f} MB')
!ls -la $OUT_DIR

## ⏸ Report back

Send `staged/b0_classifier_scale_results.zip` (or paste `classifier_scale_test.md`). On the Mac it goes into
`results/`, and the rescaled logits in `classifier_scale_logits.parquet` are merged into the cache under their
own `model_version` tags.

Check before trusting the numbers:
- **Parity line**: 2,737 of 2,737 frames matched, max |Δlogit| small (GPU vs the GPU that built the cache).
- **Unscaled held-out row** equals `results/classifier_c2c12.md`: 5.0% normal→normal, 91.8% contamination recall.